# Notebook 01: Exploratory Data Analysis & Feature Audit
### Predicting 30-Day Hospital Readmission in Patients With Diabetes
**Paper Reference:** Oluwabukola G. Emi-Johnson & Kwame J. Nkrumah (*Cureus*, April 17, 2025, DOI: 10.7759/cureus.82437)

This notebook audits the raw **UCI Diabetes 130-US Hospitals dataset** (101,766 encounters, 50 attributes, 1999–2008).
We analyze missing data encodings, class distributions, patient repeated hospitalizations, and clinical feature properties.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load raw dataset keeping 'None' as string category rather than default NaN
df = pd.read_csv('../diabetic_data.csv', keep_default_na=False)
print(f'Dataset Dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns')

## 1. Target Variable Distribution and Class Imbalance
The raw `readmitted` attribute contains `<30`, `>30`, and `NO`.
The research objective focuses on acute 30-day readmissions (<30 $\to$ 1, others $\to$ 0).

In [ ]:
raw_counts = df['readmitted'].value_counts()
raw_pcts = df['readmitted'].value_counts(normalize=True) * 100
target_summary = pd.DataFrame({'Encounter_Count': raw_counts, 'Percentage': raw_pcts.round(2)})
print('--- Raw Readmission Distribution ---')
print(target_summary)

# Binary Target Definition
y = (df['readmitted'] == '<30').astype(int)
pos_count = (y == 1).sum()
neg_count = (y == 0).sum()
print(f'\nBinary Class 1 (<30d): {pos_count:,} ({pos_count/len(y)*100:.2f}%)')
print(f'Binary Class 0 (>=30d / NO): {neg_count:,} ({neg_count/len(y)*100:.2f}%)')
print(f'Imbalance Ratio (Neg:Pos): {neg_count / pos_count:.2f} : 1')

## 2. Missing Value Representations and Clinical Distinctions
- `?` represents true missing values in categorical fields.
- In `A1Cresult` and `max_glu_serum`, `'None'` indicates **the lab test was not ordered** during hospitalization, which is an informative clinical status.

In [ ]:
missing_summary = []
for col in df.columns:
    q_count = (df[col] == '?').sum()
    if q_count > 0:
        missing_summary.append({
            'Feature': col,
            'Question_Marks': q_count,
            'Percentage': round((q_count / len(df)) * 100, 2)
        })
pd.DataFrame(missing_summary).sort_values(by='Question_Marks', ascending=False)

## 3. Patient Encounters & Overlap
71,518 unique patients account for 101,766 encounters.

In [ ]:
n_encounters = df['encounter_id'].nunique()
n_patients = df['patient_nbr'].nunique()
multi_visits = (df['patient_nbr'].value_counts() > 1).sum()
print(f'Total Encounters: {n_encounters:,}')
print(f'Unique Patients: {n_patients:,}')
print(f'Patients with Multiple Encounters: {multi_visits:,} ({multi_visits/n_patients*100:.2f}%)')

## 4. Key Predictor Visualizations
Visualizing continuous utilization metrics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(df['time_in_hospital'], bins=14, kde=False, ax=axes[0, 0], color='#1f77b4')
axes[0, 0].set_title('Length of Stay (Days)')
sns.histplot(df['num_medications'], bins=30, kde=True, ax=axes[0, 1], color='#ff7f0e')
axes[0, 1].set_title('Number of Medications')
sns.histplot(df['num_lab_procedures'], bins=30, kde=True, ax=axes[1, 0], color='#2ca02c')
axes[1, 0].set_title('Number of Lab Procedures')
sns.countplot(x=df['number_inpatient'].clip(upper=6), ax=axes[1, 1], color='#9467bd')
axes[1, 1].set_title('Prior Inpatient Admissions (Capped at 6)')
plt.tight_layout()
plt.show()